In [1]:
print('ok')

ok


In [1]:
!pip install -U langchain langchain-community langchain-core langchain-openai langchain-pinecone langchain-huggingface pypdf python-dotenv pinecone-client sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.4/170.4 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 46.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.8/573.8 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.4/146.4 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.7/402.7 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 862.5/862.5 kB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.3/259.3 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
from google.colab import files

# 1. Create the Data directory on the remote Colab server
os.makedirs('Data', exist_ok=True)

# 2. Move into the folder, prompt for upload, then move back out
os.chdir('Data')
print("Click the button below to upload your medical PDFs:")
uploaded = files.upload()
os.chdir('..')

print("\nUpload complete! The Data/ folder is ready.")

Click the button below to upload your medical PDFs:


In [2]:
import os
from dotenv import load_dotenv
from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec

# 1. Environment Loading & Validation
load_dotenv(override=True)
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_CHAT_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME")

if not all([PINECONE_API_KEY, AZURE_OPENAI_API_KEY, AZURE_OPENAI_ENDPOINT, AZURE_OPENAI_CHAT_DEPLOYMENT_NAME]):
    raise RuntimeError("Missing one or more required environment variables (Pinecone or Azure). Check your .env file.") 

# 2. Corrected LangChain Imports
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# ---> THE CORRECTED CLASSIC IMPORTS <---
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

# 3. Load & Split Data
def load_pdf_file(data_path):
    loader = DirectoryLoader(data_path, glob="*.pdf", loader_cls=PyPDFLoader)
    return loader.load()

def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    return text_splitter.split_documents(extracted_data)

print("Loading and splitting PDF data...")
extracted_data = load_pdf_file(data_path='Data/')
text_chunks = text_split(extracted_data)
print(f"Total Text Chunks: {len(text_chunks)}")

# 4. Initialize Embeddings
print("Loading HuggingFace Embeddings...")
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

# 5. Safe Pinecone Index Creation
pc = Pinecone(api_key=PINECONE_API_KEY)
index_name = "medicalbot"

# Only create the index if it doesn't already exist to prevent 409 Conflict errors
if index_name not in [idx.name for idx in pc.list_indexes()]:
    print(f"Creating Pinecone index '{index_name}'...")
    pc.create_index(
        name=index_name,
        dimension=384, 
        metric="cosine", 
        spec=ServerlessSpec(cloud="aws", region="us-east-1") 
    ) 
else:
    print(f"Pinecone index '{index_name}' already exists.")

# 6. Smart Vector Store Connection
index = pc.Index(index_name)
index_stats = index.describe_index_stats()

if index_stats.get('total_vector_count', 0) == 0:
    print("Index is empty. Upserting documents... (This may take a moment)")
    docsearch = PineconeVectorStore.from_documents(
        documents=text_chunks,
        index_name=index_name,
        embedding=embeddings, 
    )
else:
    print("Index is already populated. Connecting to existing data...")
    docsearch = PineconeVectorStore.from_existing_index(
        index_name=index_name,
        embedding=embeddings
    )

retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 3})

# 7. Initialize Azure OpenAI LLM
print("Initializing Azure OpenAI...")
llm = AzureChatOpenAI(
    azure_deployment=AZURE_OPENAI_CHAT_DEPLOYMENT_NAME,
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-02-01"),
    temperature=0.4,
    max_tokens=500
)

# 8. Build Prompt & RAG Chains
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

# 9. Test Query 
query = "What is Acne?"
print(f"\nQuerying Chatbot: '{query}'")
response = rag_chain.invoke({"input": query})

print("\n--- Final Answer ---")
print(response["answer"])




/tmp/ipykernel_1855/2887746016.py:17: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader


Loading and splitting PDF data...


FileNotFoundError: Directory not found: 'Data/'

Loading embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Connecting to Pinecone...


ValueError: Pinecone API key must be provided in either `pinecone_api_key` or `PINECONE_API_KEY` environment variable

/tmp/ipykernel_6730/3249967298.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings


ModuleNotFoundError: No module named 'langchain.chains'